In [ ]:
# ============================================
# stack_experiment_2.ipynb
#
# [실험 목적]
# stack_experiment에서 비교한 임베딩 검색 결과에 리랭커(bge-reranker)를
# 추가로 적용했을 때, 그리고 BM25/하이브리드 검색을 추가했을 때 실패
# 케이스들의 정답 청크 순위가 실제로 개선되는지 확인하는 게 목적
#
# [진행 방식]
# 1. bge-reranker-v2-m3 로드, search_and_rerank()로 BGE 검색 결과를
#    재정렬해서 정답 순위 변화 측정
# 2. KURE 임베딩 기준으로도 동일하게 리랭커 적용(search_and_rerank_kure),
#    top_k_search를 30->50->1500까지 넓혀가며 그래도 안 잡히는 실패
#    케이스(failed_cases: 한국철도공사 하도급/제안서 보상, 국립인천
#    해양박물관 검수 후 의무 등)를 별도로 추림
# 3. Kiwi 형태소 분석기로 한국어 토크나이징한 뒤 BM25Okapi 인덱스 구축,
#    find_rank_bm25()로 같은 실패 케이스들의 순위 재측정
# 4. KURE 벡터 검색 + BM25를 가중합(alpha=0.5)한 하이브리드 검색
#    (hybrid_search_rank)까지 구현해서 최종 비교
#
# [알아낸 것]
# 순수 벡터 검색으로 안 잡히던 케이스들이 리랭커·BM25·하이브리드 검색
# 단계를 거치며 얼마나 개선되는지 단계별로 확인 -> 이 결과가 최종
# 파이프라인의 하이브리드 검색(KURE+BM25) 채택 여부를 판단하는 근거로
# 쓰였으며, 이후 faiss_chroma_experiment에서 "하이브리드 적용해도
# 효과가 미미하다"는 결론과 이어짐
# ============================================

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import torch
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

In [3]:
from FlagEmbedding import FlagReranker
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)
print("reranker 로드 완료")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

reranker 로드 완료


In [4]:
import sys
import types
import pickle
from pathlib import Path

src_module = types.ModuleType('src')
chunking_module = types.ModuleType('src.chunking')

class Chunk:
    pass

chunking_module.Chunk = Chunk
src_module.chunking = chunking_module
sys.modules['src'] = src_module
sys.modules['src.chunking'] = chunking_module

DATA_DIR = Path('/content/drive/MyDrive/중급 프로젝트')

with open(DATA_DIR / 'chunks.pkl', 'rb') as f:
    chunk_objects = pickle.load(f)

all_chunks_final = []
chunk_metadata_final = []

for c in chunk_objects:
    all_chunks_final.append(c.text)
    meta_info = c.metadata if c.metadata else {}
    chunk_metadata_final.append({
        '파일명': c.doc_id,
        '발주기관': meta_info.get('발주_기관', ''),
        '사업금액': meta_info.get('사업_금액', None),
        '마감일': meta_info.get('입찰_참여_마감일', ''),
    })

print(f"청크 로드 완료: {len(all_chunks_final)}개")

청크 로드 완료: 18239개


In [8]:
import faiss
import numpy as np

with open(DATA_DIR / 'bge_embeddings.pkl', 'rb') as f:
    bge_embeddings = pickle.load(f)

dim_bge = bge_embeddings.shape[1]
index_bge = faiss.IndexFlatL2(dim_bge)
index_bge.add(np.array(bge_embeddings).astype('float32'))

print(f"bge 인덱스: {index_bge.ntotal}개")

bge 인덱스: 18239개


In [9]:
from FlagEmbedding import BGEM3FlagModel
bge_model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [10]:
def search_and_rerank(query, target_keyword, target_doc_keyword, top_k_search=30):
    query_embedding = bge_model.encode([query])['dense_vecs']
    distances, indices = index_bge.search(np.array(query_embedding).astype('float32'), top_k_search)
    candidate_indices = indices[0]

    pairs = [[query, all_chunks_final[i]] for i in candidate_indices]
    scores = reranker.compute_score(pairs)

    reranked = sorted(zip(candidate_indices, scores), key=lambda x: x[1], reverse=True)

    for new_rank, (idx, score) in enumerate(reranked, 1):
        if target_doc_keyword in chunk_metadata_final[idx]['파일명'] and target_keyword in all_chunks_final[idx]:
            return new_rank
    return None

In [11]:
test_cases_all = [
    ("검찰의 아태지역 사이버범죄 교육 플랫폼 개선 사업과 국가 교육과정 정보 제공 사이트 운영 사업 중 예산이 더 큰 쪽은 어디이며 차액은 얼마인가요?", "사 업 비", "대검찰청"),
    ("한영대학교 트랙운영 학사정보시스템 고도화 사업에서 제안 관련 제출물의 수량은 어떻게 되나요?", "10부", "한영대학"),
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?", "하도급을 불허", "운행정보기록"),
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 보상 대상이 아닌가요?", "제안서 보상", "운행정보기록"),
    ("국립인천해양박물관 해양자료관리시스템 구축사가 검수 후 부담해야 할 유지관리·교육 의무", "하자보수", "국립인천해양박물관"),
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업에서 공동수급 시 최소지분율과 구성원 수 상한은 어떻게 되나요?", "지분율", "운행정보기록"),
]

for query, keyword, doc_keyword in test_cases_all:
    rerank_result = search_and_rerank(query, keyword, doc_keyword)
    print(f"{query[:35]}... | reranker 적용 후 순위: {rerank_result}")

검찰의 아태지역 사이버범죄 교육 플랫폼 개선 사업과 국가 교육과... | reranker 적용 후 순위: 2
한영대학교 트랙운영 학사정보시스템 고도화 사업에서 제안 관련 제... | reranker 적용 후 순위: 4
한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가... | reranker 적용 후 순위: None
한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 ... | reranker 적용 후 순위: None
국립인천해양박물관 해양자료관리시스템 구축사가 검수 후 부담해야 ... | reranker 적용 후 순위: None
한국철도공사 운행정보기록 자동분석시스템 개량 사업에서 공동수급 ... | reranker 적용 후 순위: None


In [12]:
with open(DATA_DIR / 'kure_embeddings.pkl', 'rb') as f:
    kure_embeddings = pickle.load(f)

dim_kure = kure_embeddings.shape[1]
index_kure = faiss.IndexFlatL2(dim_kure)
index_kure.add(np.array(kure_embeddings).astype('float32'))

print(f"KURE 인덱스: {index_kure.ntotal}개")

KURE 인덱스: 18239개


In [15]:
import torch

from sentence_transformers import SentenceTransformer
kure_model = SentenceTransformer('nlpai-lab/KURE-v1', model_kwargs={'torch_dtype': torch.float16})
print(kure_model.device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/807 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

cuda:0


In [16]:
def search_and_rerank_kure(query, target_keyword, target_doc_keyword, top_k_search=50):
    query_embedding = kure_model.encode([query])
    distances, indices = index_kure.search(np.array(query_embedding).astype('float32'), top_k_search)
    candidate_indices = indices[0]

    pairs = [[query, all_chunks_final[i]] for i in candidate_indices]
    scores = reranker.compute_score(pairs)

    reranked = sorted(zip(candidate_indices, scores), key=lambda x: x[1], reverse=True)

    for new_rank, (idx, score) in enumerate(reranked, 1):
        if target_doc_keyword in chunk_metadata_final[idx]['파일명'] and target_keyword in all_chunks_final[idx]:
            return new_rank
    return None

In [17]:
for query, keyword, doc_keyword in test_cases_all:
    kure_rerank_result = search_and_rerank_kure(query, keyword, doc_keyword)
    print(f"{query[:35]}... | KURE+reranker 적용 후 순위: {kure_rerank_result}")

검찰의 아태지역 사이버범죄 교육 플랫폼 개선 사업과 국가 교육과... | KURE+reranker 적용 후 순위: 2
한영대학교 트랙운영 학사정보시스템 고도화 사업에서 제안 관련 제... | KURE+reranker 적용 후 순위: 4
한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가... | KURE+reranker 적용 후 순위: None
한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 ... | KURE+reranker 적용 후 순위: None
국립인천해양박물관 해양자료관리시스템 구축사가 검수 후 부담해야 ... | KURE+reranker 적용 후 순위: None
한국철도공사 운행정보기록 자동분석시스템 개량 사업에서 공동수급 ... | KURE+reranker 적용 후 순위: None


In [18]:
def search_and_rerank_kure_wide(query, target_keyword, target_doc_keyword, top_k_search=1500):
    query_embedding = kure_model.encode([query])
    distances, indices = index_kure.search(np.array(query_embedding).astype('float32'), top_k_search)
    candidate_indices = indices[0]

    pairs = [[query, all_chunks_final[i]] for i in candidate_indices]
    scores = reranker.compute_score(pairs, batch_size=64)

    reranked = sorted(zip(candidate_indices, scores), key=lambda x: x[1], reverse=True)

    for new_rank, (idx, score) in enumerate(reranked, 1):
        if target_doc_keyword in chunk_metadata_final[idx]['파일명'] and target_keyword in all_chunks_final[idx]:
            return new_rank
    return None

In [19]:
failed_cases = [
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?", "하도급을 불허", "운행정보기록"),
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 보상 대상이 아닌가요?", "제안서 보상", "운행정보기록"),
    ("국립인천해양박물관 해양자료관리시스템 구축사가 검수 후 부담해야 할 유지관리·교육 의무", "하자보수", "국립인천해양박물관"),
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업에서 공동수급 시 최소지분율과 구성원 수 상한은 어떻게 되나요?", "지분율", "운행정보기록"),
]

for query, keyword, doc_keyword in failed_cases:
    result = search_and_rerank_kure_wide(query, keyword, doc_keyword)
    print(f"{query[:35]}... | top-1500+rerank 순위: {result}")

Compute Scores: 100%|██████████| 24/24 [00:31<00:00,  1.31s/it]


한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가... | top-1500+rerank 순위: 273


Compute Scores: 100%|██████████| 24/24 [00:28<00:00,  1.18s/it]


한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 ... | top-1500+rerank 순위: 80


Compute Scores: 100%|██████████| 24/24 [00:29<00:00,  1.22s/it]


국립인천해양박물관 해양자료관리시스템 구축사가 검수 후 부담해야 ... | top-1500+rerank 순위: 49


Compute Scores: 100%|██████████| 24/24 [00:33<00:00,  1.38s/it]


한국철도공사 운행정보기록 자동분석시스템 개량 사업에서 공동수급 ... | top-1500+rerank 순위: 1310


In [21]:
from kiwipiepy import Kiwi
kiwi = Kiwi()

def tokenize_korean(text):
    return [token.form for token in kiwi.tokenize(text)]

In [22]:
from rank_bm25 import BM25Okapi
from tqdm import tqdm

tokenized_chunks = [tokenize_korean(c) for c in tqdm(all_chunks_final)]
bm25 = BM25Okapi(tokenized_chunks)
print("BM25 인덱스 생성 완료")

100%|██████████| 18239/18239 [06:18<00:00, 48.23it/s]


BM25 인덱스 생성 완료


In [23]:
def find_rank_bm25(query, target_keyword, target_doc_keyword, bm25, tokenize_fn, all_chunks, chunk_metadata):
    tokenized_query = tokenize_fn(query)
    scores = bm25.get_scores(tokenized_query)
    ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)

    for rank, idx in enumerate(ranked_indices, 1):
        if target_doc_keyword in chunk_metadata[idx]['파일명'] and target_keyword in all_chunks[idx]:
            return rank
    return None

In [24]:
bm25_test_cases = [
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?", "하도급을 불허", "운행정보기록"),
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 보상 대상이 아닌가요?", "제안서 보상", "운행정보기록"),
    ("국립인천해양박물관 해양자료관리시스템 구축사가 검수 후 부담해야 할 유지관리·교육 의무", "하자보수", "국립인천해양박물관"),
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업에서 공동수급 시 최소지분율과 구성원 수 상한은 어떻게 되나요?", "지분율", "운행정보기록"),
]

for query, keyword, doc_keyword in bm25_test_cases:
    bm25_rank = find_rank_bm25(query, keyword, doc_keyword, bm25, tokenize_korean, all_chunks_final, chunk_metadata_final)
    print(f"{query[:35]}... | BM25 순위: {bm25_rank}")

한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가... | BM25 순위: 346
한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 ... | BM25 순위: 143
국립인천해양박물관 해양자료관리시스템 구축사가 검수 후 부담해야 ... | BM25 순위: 24
한국철도공사 운행정보기록 자동분석시스템 개량 사업에서 공동수급 ... | BM25 순위: 34


In [25]:
def hybrid_search_rank(query, target_keyword, target_doc_keyword, kure_model, index_kure, bm25, tokenize_fn, all_chunks, chunk_metadata, alpha=0.5, top_k=2000):
    query_embedding = kure_model.encode([query])
    dense_distances, dense_indices = index_kure.search(np.array(query_embedding).astype('float32'), top_k)

    max_dist = dense_distances[0].max()
    dense_scores = {idx: 1 - (dist / max_dist) for idx, dist in zip(dense_indices[0], dense_distances[0])}

    tokenized_query = tokenize_fn(query)
    bm25_scores_all = bm25.get_scores(tokenized_query)
    max_bm25 = bm25_scores_all.max() if bm25_scores_all.max() > 0 else 1

    combined_scores = {}
    for idx in dense_indices[0]:
        dense_s = dense_scores.get(idx, 0)
        bm25_s = bm25_scores_all[idx] / max_bm25
        combined_scores[idx] = alpha * dense_s + (1 - alpha) * bm25_s

    for idx in range(len(all_chunks)):
        if idx not in combined_scores:
            bm25_s = bm25_scores_all[idx] / max_bm25
            if bm25_s > 0:
                combined_scores[idx] = (1 - alpha) * bm25_s

    ranked = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)

    for rank, (idx, score) in enumerate(ranked, 1):
        if target_doc_keyword in chunk_metadata[idx]['파일명'] and target_keyword in all_chunks[idx]:
            return rank
    return None

In [26]:
hybrid_test_cases = [
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?", "하도급을 불허", "운행정보기록"),
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 보상 대상이 아닌가요?", "제안서 보상", "운행정보기록"),
    ("국립인천해양박물관 해양자료관리시스템 구축사가 검수 후 부담해야 할 유지관리·교육 의무", "하자보수", "국립인천해양박물관"),
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업에서 공동수급 시 최소지분율과 구성원 수 상한은 어떻게 되나요?", "지분율", "운행정보기록"),
]

for query, keyword, doc_keyword in hybrid_test_cases:
    hybrid_rank = hybrid_search_rank(query, keyword, doc_keyword, kure_model, index_kure, bm25, tokenize_korean, all_chunks_final, chunk_metadata_final)
    print(f"{query[:35]}... | 하이브리드 순위: {hybrid_rank}")

한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가... | 하이브리드 순위: 428
한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 ... | 하이브리드 순위: 135
국립인천해양박물관 해양자료관리시스템 구축사가 검수 후 부담해야 ... | 하이브리드 순위: 32
한국철도공사 운행정보기록 자동분석시스템 개량 사업에서 공동수급 ... | 하이브리드 순위: 66
